In [1]:
"""属性库整理"""

from pathlib import Path
import json
import sys

paths = [
    Path().resolve().parent / "utility/biz_utils",
    Path().resolve().parent.parent / "utility/biz_utils",
]
for path in paths:
    if path.exists() and str(path) not in sys.path:
        sys.path.append(str(path))

from properties_tools import properties_organize, properties_organize_simple, properties_organize_simple_v2

def properties_organize_func():
    json_path = Path().resolve() / "properties_remote.json"
    properties_data = json.loads(json_path.read_text())
    properties_organize_result = properties_organize(properties_data)

    json_path = Path().resolve() / "properties_remote_organize.json"
    properties_organize_json = json.dumps(properties_organize_result, indent=4, ensure_ascii=False)
    json_path.write_text(properties_organize_json)
    print(json_path)

    json_path = Path().resolve() / "properties_remote_simple_v1.json"
    properties_simple = properties_organize_simple(properties_organize_result)

    properties_simple_json = json.dumps(properties_simple, indent=4, ensure_ascii=False)
    json_path.write_text(properties_simple_json)
    print(json_path)

    json_path = Path().resolve() / "properties_remote_simple_v2.json"
    properties_simple = properties_organize_simple_v2(properties_organize_result)

    properties_simple_json = json.dumps(properties_simple, indent=4, ensure_ascii=False)
    json_path.write_text(properties_simple_json)
    print(json_path)

# properties_organize_func()

In [2]:
"""按 CategoryID 拉取属性库"""

import requests
import time
from pathlib import Path
import pyperclip
import json
import os
import gzip
import base64
from io import BytesIO

config_path = Path("localtest_config.json").resolve().read_text()
config = json.loads(config_path)
categoryId = config.get("categoryId") or pyperclip.paste()

assert categoryId.isdigit(), "请复制cid到剪贴板"
print("categoryId =", categoryId)
# host = "pre-adc-pc.alibaba-inc.com"   # 预发环境
host = "adc-pc.alibaba-inc.com"  # 生产环境

url = f"http://{host}/task/queryGamePropertyByCid"
payload = {"cid": categoryId}
response = requests.post(url, json=payload)

def ungzip_text(text: str) -> str:
    if not text or text.isspace():
        return ""
    with BytesIO(base64.b64decode(text)) as bytes_in:
        with gzip.GzipFile(fileobj=bytes_in, mode='rb') as gzip_in:
            decompressed_bytes = gzip_in.read()
            return decompressed_bytes.decode('utf-8')

json_path = Path().resolve() / "properties_remote.json"
if json_path.exists(): os.remove(json_path)

propertyValuesCompressed = json.loads(response.json()["data"])["result"]["gamePropertiesCompressed"]
json_str = json.dumps(json.loads(ungzip_text(propertyValuesCompressed)), indent=4, ensure_ascii=False)
json_path.write_text(json_str)
print(json_path)
properties_organize_func()

categoryId = 1758104636136879
/Users/vip/adc-sc/5799_steam/properties_remote.json
/Users/vip/adc-sc/5799_steam/properties_remote_organize.json
/Users/vip/adc-sc/5799_steam/properties_remote_simple_v1.json
/Users/vip/adc-sc/5799_steam/properties_remote_simple_v2.json


In [ ]:
"""按 TaskID 拉取属性库"""

import requests
import time
from pathlib import Path
import pyperclip
import json
import os

task_id = pyperclip.paste()

assert task_id.isdigit(), "请复制任务id到剪贴板"
print("task_id =", task_id)
# host = "pre-adc-pc.alibaba-inc.com"   # 预发环境
host = "adc-pc.alibaba-inc.com"  # 生产环境

url = f"http://{host}/task/getGamePropertiesByTaskId"
payload = {"taskId": task_id, "timestamp": int(time.time() * 1000)}
response = requests.post(url, json=payload)

json_path = Path().resolve() / "properties_remote.json"
if json_path.exists(): os.remove(json_path)
json_str = json.dumps(json.loads(response.json()["data"]["gameProperties"]), indent=4, ensure_ascii=False)
json_path.write_text(json_str)
print(json_path)
properties_organize_func()

In [23]:
"""本地采集数据"""

import json
import shelve
from pathlib import Path
from pprint import pprint
import yaml

shelve_path = str(Path().resolve() / "properties_data")
with shelve.open(shelve_path) as properties_data:
    properties_dict = dict(properties_data)
    properties_json = json.dumps(properties_dict, indent=4, ensure_ascii=False)
    (Path().resolve() / "properties_data_local.json").write_text(properties_json)
    print(yaml.dump(properties_dict, allow_unicode=True, sort_keys=False).strip())

荣耀水晶数: '0'
VIP等级: '8'
小兵数: 0
天幕数: 1
星传说皮肤数: 0
天幕: []


In [24]:
"""线上回传属性解压"""

import gzip
import base64
from io import BytesIO
import json
import pyperclip
from pathlib import Path

def gzip_text(text: str) -> str:
    if not text or text.isspace():
        return ""

    with BytesIO() as bytes_out:
        with gzip.GzipFile(fileobj=bytes_out, mode='wb') as gzip_out:
            gzip_out.write(text.encode('utf-8'))
        compressed_bytes = bytes_out.getvalue()
        return base64.b64encode(compressed_bytes).decode('ascii')

def ungzip_text(text: str) -> str:
    if not text or text.isspace():
        return ""
    with BytesIO(base64.b64decode(text)) as bytes_in:
        with gzip.GzipFile(fileobj=bytes_in, mode='rb') as gzip_in:
            decompressed_bytes = gzip_in.read()
            return decompressed_bytes.decode('utf-8')

# propertyValuesCompressed = pyperclip.paste()
propertyValuesCompressed = json.loads(pyperclip.paste())["propertyValuesCompressed"]
json_str = json.dumps(json.loads(ungzip_text(propertyValuesCompressed)), indent=4, ensure_ascii=False)
(Path().resolve() / "properties_callback_online.json").write_text(json_str)

1955

In [ ]:
"""回传属性分析"""

from pathlib import Path
import json
from pprint import pprint

filePath = Path().resolve() / "properties_callback_online.json"
# filePath = Path().resolve() / "properties_callback_local.json"
properties_callback_online = json.loads(filePath.read_text())

properties_callback_analyze = {}
for level0_item in properties_callback_online:
    level0_name = level0_item["propertyName"]
    for level1_item in level0_item.get("gameProperties", []):
        level1_name = level1_item["value"]
        level1_data = {"名称": level1_name}

        # 二级属性
        for level2_item in level1_item["childProperties"]:
            level1_data[level2_item["childName"]] = level2_item["value"]

        properties_callback_analyze.setdefault(level0_name, []).append(level1_data)

level2_keys = {
    "角色服装": ["名称", "品质", "部位"],
    "枪械皮肤": ["名称", "品质", "枪械", "种类"],
    "载具皮肤": ["名称", "品质", "载具类型"],
}
for key, values in properties_callback_analyze.items():
    print(key, len(values))
    for i in values:
        if set(level2_keys[key]).difference(i):
            print(i)


In [ ]:
"""ADC属性库更新"""

from pathlib import Path
import json

propertiesUpdateTemplates = {}
json_path = Path().resolve() / "propertiesUpdateTemplates.json"
if json_path.exists():
    json_data = json.loads(json_path.read_text())
    for property_item in json_data:
        propertiesUpdateTemplates[property_item["propertyName"]] = property_item

propertiesUpdateTemplates

In [ ]:
"""交易猫属性库更新"""

from time import sleep
import json
from pathlib import Path
import sys

import requests

paths = [
    Path().resolve().parent / "utility/biz_utils",
    Path().resolve().parent.parent / "utility/biz_utils",
]
for path in paths:
    if path.exists() and str(path) not in sys.path:
        sys.path.append(str(path))
from biz_tools import data_pagination, gzip_text

def get_newgameid_by_oldgameid(old_game_id):
    host = 'adc-pc.alibaba-inc.com'
    url = f"http://{host}/task/getNewGameIdByOldGameId"
    payload = {
        "oldGameId": old_game_id
    }
    response = requests.post(url, json=payload)
    response_json = response.json()
    if newGameId := json.loads(response_json.get("data")).get("result"):
        return newGameId

def upload_to_jym(newGameId, data):
    [x.pop("imageUrl", None) for x in data]
    json_data = json.dumps(data, ensure_ascii=False)
    print(json_data)
    json_data_zip = gzip_text(json_data)

    host = 'adc-pc.alibaba-inc.com'
    url = "http://{}/task/syncGameProperty".format(host)
    payload = {
        "gameId": newGameId,
        "gamePropertiesCompressed": json_data_zip
    }
    response = requests.post(url, json=payload)
    return response.json()

properties_update_list = []

oldGameId = 5799
# oldGameId = 8788
newGameId = get_newgameid_by_oldgameid(oldGameId)
if not newGameId:
    print(f"获取 newGameID 失败")
else:
    for data in data_pagination(properties_update_list, 20):
        print(f"提交数据: {len(data)}条", )
        print(f"{newGameId} 上传结果:", upload_to_jym(newGameId, data))

提交数据: 13条
[{"childNodes": [{"propertyName": "品质", "propertyValueName": "精英"}, {"propertyName": "枪械", "propertyValueName": "Mk47 Mutant"}, {"propertyName": "种类", "propertyValueName": "突击步枪"}], "propertyName": "枪械皮肤", "propertyValueName": "行星保卫队-MK47MUTANT"}, {"childNodes": [{"propertyName": "品质", "propertyValueName": "史诗"}, {"propertyName": "枪械", "propertyValueName": "SCAR-L"}, {"propertyName": "种类", "propertyValueName": "突击步枪"}], "propertyName": "枪械皮肤", "propertyValueName": "行星保卫队-SCAR-L"}, {"childNodes": [{"propertyName": "品质", "propertyValueName": "传说"}, {"propertyName": "枪械", "propertyValueName": "Kar98k"}, {"propertyName": "种类", "propertyValueName": "狙击枪"}, {"propertyName": "成长", "propertyValueName": "LV1"}, {"propertyName": "成长", "propertyValueName": "LV2"}, {"propertyName": "成长", "propertyValueName": "LV3"}, {"propertyName": "成长", "propertyValueName": "LV4"}, {"propertyName": "成长", "propertyValueName": "LV5"}, {"propertyName": "成长", "propertyValueName": "LV6"}, {"propertyName": "

In [22]:
"""清理多余图片"""

from pathlib import Path
import os
from pprint import pprint
import re

gameDir = Path().resolve()
for tasktypedir in gameDir.glob(f"{gameDir.name}*"):
    py_content_list = []
    for py_path_px in tasktypedir.glob("*.py"):
        py_content_list.append(py_path_px.read_text())
    py_content = "".join(py_content_list)

    pic_list = tasktypedir.glob("tpl*.png")
    rm_pic_list = list(filter(lambda pic_path: not re.findall(pic_path.name, py_content), pic_list))
    if rm_pic_list:
        for pic_path in rm_pic_list: os.remove(pic_path)
        print(f"{tasktypedir.stem} 清理 {len(rm_pic_list)} 张图片")

In [ ]:
"""线上业务串流"""

import time
from pathlib import Path
import os

import yaml
from jsonpath import jsonpath
import requests


class OnlineTest:
    def __init__(self):
        taskType = 2  # 发布2，验号3
        # host = "test-adc-pc.alibaba.net"   # 日常环境
        host = "pre-adc-pc.alibaba-inc.com"  # 预发环境

        self.timeStamp = lambda: str(int(time.time() * 10 ** 3))
        self.submitTask_url = f"https://{host}/task/submitTask"
        self.taskstream_url = f"https://{host}/taskstream?taskId="
        self.params = dict(gameId=13765, taskType=taskType, gamePlatform="shiyue", taskStatus=0, deviceInfo="", scriptId="", resultInfo="")
        self.session = requests.session()
        headers = {"User-Agent": "Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Mobile Safari/537.36"}
        self.session.headers.update(headers)

    def __call__(self, *args, **kwds):
        self.params.update(userAccount="13581974227")
        self.params.update(password="xz123456")
        self.params.update(gameArea="玉兔庆新春")
        self.params.update(game_area="玉兔庆新春")
        self.params.update(bizId=f"test{self.timeStamp()}")
        res = self.session.get(self.submitTask_url, params=self.params)
        print(res.json())
        # if taskId := jsonpath(res.json(), "$..taskId"):

OnlineTest()()

In [ ]:
submitTask_url = f"https://{host}/task/submitTask"

In [ ]:
"""文件重命名"""

import time
from glob import glob
import os

OPJ = os.path.join
OP = os.path

timeStamp = lambda: str(int(time.time() * 10 ** 6))


class ScreenshotRename:
    def __init__(self):
        self.filesList = glob(OPJ("/Users/vip/androidLogResult", "adc_result_*_$*.jpg"))

    def filename_split(self, old_filename):
        filename = OP.splitext(old_filename)[0]
        filename_split_list = filename.split("_")
        s1, s2 = filename_split_list[-2:]
        serial = int(s1) if s1.isdigit() else 0
        id_str = s2.strip("$")
        id_int = int(id_str) if id_str.isdigit() else 0
        new_filename = f"{'_'.join(filename_split_list[:-1])}"
        return old_filename, new_filename, id_int, serial

    def online(self, filesList_order):
        for old_filename, new_filename, *args in filesList_order:
            ext = OP.splitext(old_filename)[1]
            new_filename += f"_{timeStamp()}{ext}"
            os.rename(old_filename, new_filename)
            time.sleep(0.1)

    def offline(self, filesList_order):
        for old_filename, new_filename, id_int, serial in filesList_order:
            path, name = OP.split(new_filename)
            name_split = name.split("_")
            if name_split[-1].isdigit(): name = "_".join(name_split[:-1])
            ext = OP.splitext(old_filename)[1]
            new_filename = OPJ(path, f"{id_int}_{serial}_{name}{ext}")
            os.rename(old_filename, new_filename)

    def __call__(self, *args, **kwds):
        filesList_order = list(map(self.filename_split, self.filesList))
        filesList_order.sort(key=lambda item: item[-2:])
        self.offline(filesList_order)
        # self.online(filesList_order)


ScreenshotRename()()

In [ ]:
"""JSONC转YAML"""

# from jsonc_parser.parser import JsoncParser
# import yaml

# json_data = JsoncParser.parse_file(Path().resolve() / "onlinetest_config.jsonc")
# yaml_path = Path().resolve() / "onlinetest_config.yaml"
# with open(yaml_path, mode='w', encoding="utf8") as yaml_file:
#     yaml.dump(json_data, yaml_file, allow_unicode=True, sort_keys=False, indent=4)

In [ ]:
"""JSON转YAML"""

import json
import yaml
from pprint import pprint
from pathlib import Path

json_path = Path().resolve() / "rectify.json"
yaml_path = Path().resolve() / "rectify.yaml"

with (open(json_path, mode="r", encoding="utf8") as json_file,
      open(yaml_path, mode="w", encoding="utf8") as yaml_file):
    json_data = json.load(json_file)
    yaml.dump(json_data, yaml_file, allow_unicode=True, sort_keys=False, indent=4)

with open(yaml_path, mode='r', encoding="utf8") as yaml_file:
    pprint(yaml.load(yaml_file, Loader=yaml.FullLoader))

In [ ]:
# Scrcpy 窗口缩放计算
import math

for i in range(900, 1000):
    a = 1350 * i / 2400
    if not math.modf(a)[0]:
        print(i, int(a))

In [ ]:
# 三角函数
import math

def Joystick(centre, radius, angle):
    x = centre[0] + radius * math.cos(math.radians(angle))
    y = centre[1] + radius * math.sin(math.radians(angle))
    print(f"x={int(x)}, y={int(y)}")


Joystick((100, 100), 50, -10)

In [ ]:
from enum import Enum


class Weekday(Enum):
    monday = 1
    tuesday = 2
    wednesday = 3
    thirsday = 4
    friday = 5
    saturday = 6
    sunday = 7


print(Weekday.wednesday == "wednesday")  # Weekday.wednesday
print(Weekday.wednesday)  # <enum 'Weekday'>
print(Weekday.wednesday.name)  # wednesday
print(Weekday.wednesday.value)

isinstance(Weekday.wednesday, Enum)

In [ ]:
"""属性数据后台批量操作"""

# 拉取属性数据列表
import requests
import json

# host = "test-adc-pc.alibaba.net"   # 日常环境
# host = "pre-adc-pc.alibaba-inc.com"  # 预发环境
host = "adc-pc.alibaba-inc.com"  # 生产环境

session = requests.session()
session.verify = False
requests.packages.urllib3.disable_warnings()  # type: ignore


# 属性数据过滤
def property_filter(x):
    if x["game_id"] == game_id and \
            (x["id"] <= id_max or id_max == 0) and \
            x["status"] == 0 and x["id"] not in exclude:
        json_data = json.loads(x["delta_game_data"])
        propertyName = json_data[0]["propertyName"]
        propertyValueName = json_data[0]["propertyValueName"]
        print(f"{x['id']} {propertyName}: {propertyValueName}")
        return True


game_id_list = [
    # "13765",
    # "15574",
    # "2717",
    # "6586",
    "8788",
]
id_max = 0
exclude = [
    16377
]

dataIdList = []
for index, game_id in enumerate(game_id_list):
    url = f"https://{host}/task/getGamePropertyDataList"
    payload = {"current": 1, "pageSize": 100, "game_id": game_id, "status": "0"}
    response = session.post(url, json=payload)
    propertyDataList = response.json()
    data = propertyDataList["result"]["data"]
    dataIdList = sorted(map(lambda x: x["id"], (filter(property_filter, data["content"]))), reverse=True)
    if not data['total']: continue
    if dataIdList:
        print(f"{game_id} 总数：{len(dataIdList)}")
        break

In [ ]:
# 更新属性数据后台状态
from time import sleep

url = f"https://{host}/task/updateGamePropertyDataStatus"
for data_id in dataIdList:
    payload = {"id": data_id, "status": 2}
    response = session.post(url, json=payload).json()["result"]["data"]
    print(f"id: {response['id']}, status: {response['status']}")
    sleep(0.25)

In [ ]:
"""屏蔽路径列表"""

from glob import glob
from pathlib import Path
import os
import json
import pyperclip
import re


def select_numbers(string):
    pieces = re.compile(r"(\d+)").split(string)
    pieces[1::2] = map(int, pieces[1::2])
    return pieces


exclude_list = [
    "8788",
    "13765",
    "15574",
    "6586",
    "10025",
    "2717",
    "9207",
    "25398",
    "ios_23016",
    "ios_11446",
    "9999013",
    "9999017",
    "9999018",
    "5799",
    "2416"
]

gameID_list = []
adc_path = Path().resolve().parent.parent
for path in glob(str(adc_path / "*")):
    if Path(path).is_dir():
        gameID = os.path.split(path)[1].split("_")[0]
        if gameID.isdigit():
            gameID_list.append(gameID)

for path in glob(str(adc_path / "iOS" / "*")):
    if Path(path).is_dir():
        gameID = os.path.split(path)[1].split("_")[1]
        if gameID.isdigit():
            gameID_list.append(f"ios_{gameID}")

gameID_list = sorted(set(gameID_list), key=select_numbers)

gameID_list = dict(map(lambda x: (f"**/{x}_*", True), filter(lambda x: x not in exclude_list, gameID_list)))
pyperclip.copy(json.dumps(gameID_list, indent=4).strip("{").strip("}"))

In [ ]:
"""上下文"""

from contextlib import contextmanager

@contextmanager
def my_context():
    # 进入上下文前的操作
    print('entering context')
    try:
        yield  # 生成器函数的 `yield` 语句之前的代码作为上文，之后的代码作为下文
    finally:
        # 离开上下文后的操作
        print('exiting context')


# 使用上下文管理器
with my_context():
    print('inside context')

In [ ]:
"""文件哈希值"""

import hashlib
from pathlib import Path

file_path = Path("/Users/vip/Downloads/tpl20250923161637.png").resolve()
file_byte = file_path.read_bytes()
print(hashlib.sha1(file_byte).hexdigest())

file_path = Path("/Users/vip/Downloads/single.png").resolve()
file_byte = file_path.read_bytes()
print(hashlib.sha1(file_byte).hexdigest())

In [ ]:
"""JSON 格式化"""

import pyperclip
import json

string = pyperclip.paste()
string = string.replace("'", "\"").replace("True", "true").replace("False", "false")
try:
    python_dict = json.loads(string)
except Exception:
    print("Invalid JSON")
else:
    json_str = json.dumps(python_dict, indent=4, ensure_ascii=False)
    pyperclip.copy(json_str)

In [ ]:
"""抖音达人数据上传"""

import json
from pathlib import Path

import requests

isProd = 0
isPre = 0
if isProd:
    host = "lingxi-adc-task-server.alibaba-inc.com"
    scheme = "https"
elif isPre:
    host = "pre-lingxi-adc-task-server.alibaba-inc.com"
    scheme = "https"
else:
    host = "lingxi-adc-task-sever.taobao.net"
    scheme = "http"

url = f"{scheme}://{host}/game/streamer/saveDouyin"
properties_data_local = json.loads(Path("properties_data_local.json").read_text())
json_data = {
    "taskId": "9894155",
    "list": properties_data_local["抖音达人"],
}
# pprint(json_data, sort_dicts=False)

respones = requests.post(url, json=json_data)
try:
    print(respones.json())
except:
    print("请求错误")
    print(respones.text)

In [ ]:
import re

level_pattern = re.compile(r'[（\(][0-9lL]+级[）\)]')

level_pattern.sub("", "PGC2021群雄逐鹿背包(2级）")

In [ ]:
"""YAML 读取"""

import json
import yaml
from pprint import pprint
from pathlib import Path

yaml_path = Path().resolve() / "rectify.yaml"
with open(yaml_path, mode='r', encoding="utf8") as yaml_file:
    pprint(yaml.load(yaml_file, Loader=yaml.FullLoader))

In [ ]:
"""属性库排序"""

from pathlib import Path
import json
from pprint import pprint

filePath = Path().resolve() / "properties_remote.json"
gameProperties = json.loads(filePath.read_text())


def sort_key(gps):
    sort1 = -len(gps.get("propertyValue", ""))
    sort2 = gps.get("propertyValue", "")
    return sort1, sort2


for item in gameProperties:
    propertyName = item["propertyName"]
    if propertyName in [
        # "角色服装",
        "枪械皮肤",
        # "载具皮肤",
    ]:
        gameProperties = sorted(item.get("gameProperties", []), key=sort_key)

        for i in gameProperties:
            print(i["propertyValue"])

In [ ]:
"""图片重命名"""

from pathlib import Path
import os
import re
import datetime
from time import sleep

gameDir = Path("/Users/vip/adc-sc/25115_common/25115_common_index.air")
py_path_px = gameDir / "25115_common_index.py"
pythonFileContent = py_path_px.read_text()

imageList = gameDir.glob("*.jpg")
for imagePath in list(imageList):
    filename = f"{datetime.datetime.now().strftime('tpl%Y%m%d%H%M%S')}{imagePath.suffix}"
    print(imagePath.name, filename)
    imagePath.rename(gameDir / filename)
    pythonFileContent = pythonFileContent.replace(imagePath.name, filename)
    sleep(0.5)
py_path_px.write_text(pythonFileContent)